In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import joblib
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [30]:
pick=pd.read_csv('/content/drive/MyDrive/pickup_data (2).csv')

In [31]:
time_columns = ['accept_time','time_window_start','time_window_end','pickup_time','pickup_gps_time','accept_gps_time','ds']

for col in time_columns:
    pick[col] = pd.to_datetime(pick[col],errors='coerce')

In [32]:
encoder = LabelEncoder()
pick['city'] = encoder.fit_transform(pick['city'])

In [33]:
pick.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3957642 entries, 0 to 3957641
Data columns (total 21 columns):
 #   Column             Dtype         
---  ------             -----         
 0   order_id           int64         
 1   region_id          int64         
 2   city               int64         
 3   courier_id         int64         
 4   accept_time        datetime64[ns]
 5   time_window_start  datetime64[ns]
 6   time_window_end    datetime64[ns]
 7   lng                float64       
 8   lat                float64       
 9   aoi_id             int64         
 10  aoi_type           int64         
 11  pickup_time        datetime64[ns]
 12  pickup_gps_time    datetime64[ns]
 13  pickup_gps_lng     float64       
 14  pickup_gps_lat     float64       
 15  accept_gps_time    datetime64[ns]
 16  accept_gps_lng     float64       
 17  accept_gps_lat     float64       
 18  ds                 datetime64[ns]
 19  task_duration      float64       
 20  distance           float

In [34]:
pick = pick.sort_values(by='ds')

In [35]:
pick['ETA']=pick['task_duration']

In [36]:
# Extract time features
pick['accept_hour'] = pick['accept_time'].dt.hour
pick['accept_day'] = pick['accept_time'].dt.day
pick['accept_month'] = pick['accept_time'].dt.month
pick['accept_weekday'] = pick['accept_time'].dt.weekday
pick['is_weekend'] = (pick['accept_weekday'] >= 5).astype(int)

In [37]:
pick.columns

Index(['order_id', 'region_id', 'city', 'courier_id', 'accept_time',
       'time_window_start', 'time_window_end', 'lng', 'lat', 'aoi_id',
       'aoi_type', 'pickup_time', 'pickup_gps_time', 'pickup_gps_lng',
       'pickup_gps_lat', 'accept_gps_time', 'accept_gps_lng', 'accept_gps_lat',
       'ds', 'task_duration', 'distance', 'ETA', 'accept_hour', 'accept_day',
       'accept_month', 'accept_weekday', 'is_weekend'],
      dtype='object')

In [38]:
pick.drop([ 'accept_time',
       'time_window_start', 'time_window_end',
        'pickup_time', 'pickup_gps_time', 'accept_gps_time','ds'],axis=1,inplace=True)

In [39]:
pick.head()

,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,pickup_gps_lng,pickup_gps_lat,accept_gps_lng,accept_gps_lat,task_duration,distance,ETA,accept_hour,accept_day,accept_month,accept_weekday,is_weekend
434353,1717770,60,0,3953,108.71407,30.92927,2870,4,108.71306,30.928740,106.52199,29.59889,52.0,2.563062,52.0,6,5,1,6,1
2095579,286828,130,4,4760,121.42578,37.52127,9718,1,121.23895,37.512095,121.24501,37.51106,7.0,0.006148,7.0,7,5,1,6,1
1874360,1366395,123,4,6832,120.47172,37.60178,6114,1,120.47215,37.602700,120.50282,37.60819,25.0,0.031157,25.0,7,5,1,6,1
2095496,4298043,130,4,4760,121.42581,37.52121,9718,1,121.23895,37.512095,121.24501,37.51106,18.0,0.006148,18.0,7,5,1,6,1
1569406,4943114,111,4,2235,121.27105,37.55706,21355,1,121.27078,37.557750,121.24501,37.51106,31.0,0.053330,31.0,7,5,1,6,1


In [40]:
scaler = StandardScaler()
features_to_scale = ['aoi_id','task_duration','distance','ETA']
pick[features_to_scale] = scaler.fit_transform(pick[features_to_scale])

In [41]:
pick.head()

,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,pickup_gps_lng,pickup_gps_lat,accept_gps_lng,accept_gps_lat,task_duration,distance,ETA,accept_hour,accept_day,accept_month,accept_weekday,is_weekend
434353,1717770,60,0,3953,108.71407,30.92927,-1.292602,4,108.71306,30.928740,106.52199,29.59889,-0.808711,7.457533,-0.808711,6,5,1,6,1
2095579,286828,130,4,4760,121.42578,37.52127,-0.326528,1,121.23895,37.512095,121.24501,37.51106,-1.360169,-0.282934,-1.360169,7,5,1,6,1
1874360,1366395,123,4,6832,120.47172,37.60178,-0.834959,1,120.47215,37.602700,120.50282,37.60819,-1.139586,-0.207223,-1.139586,7,5,1,6,1
2095496,4298043,130,4,4760,121.42581,37.52121,-0.326528,1,121.23895,37.512095,121.24501,37.51106,-1.225368,-0.282934,-1.225368,7,5,1,6,1
1569406,4943114,111,4,2235,121.27105,37.55706,1.315149,1,121.27078,37.557750,121.24501,37.51106,-1.066058,-0.140102,-1.066058,7,5,1,6,1


In [42]:
pick=pick.head(100000)

In [43]:
X=pick.drop(['ETA'],axis=1)
y=pick['ETA']

In [44]:
# Split into 60% train, 20% validation, 20% test
train_size = int(0.6 * len(pick))
valid_size = int(0.2 * len(pick))

X_train, y_train = X[:train_size], y[:train_size]
X_valid, y_valid = X[train_size:train_size+valid_size], y[train_size:train_size+valid_size]
X_test, y_test = X[train_size+valid_size:], y[train_size+valid_size:]

In [46]:
pick.shape

(100000, 20)

In [47]:
# Initialize and Train Gradient Boosting Regressor
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
gbr.fit(X_train, y_train)

# Predictions
y_pred_gbr = gbr.predict(X_test)

# Evaluation Metrics for GBR
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
mse_gbr = mean_squared_error(y_test, y_pred_gbr)
rmse_gbr = mse_gbr ** 0.5
r2_gbr = r2_score(y_test, y_pred_gbr)

print("Gradient Boosting Regressor Performance:")
print(f"MAE: {mae_gbr}")
print(f"MSE: {mse_gbr}")
print(f"RMSE: {rmse_gbr}")
print(f"R² Score: {r2_gbr}")

Gradient Boosting Regressor Performance:
MAE: 0.00011361736407982712
MSE: 8.060721902411702e-08
RMSE: 0.0002839141050108589
R² Score: 0.9999998959695738


In [48]:
pick.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 434353 to 2598060
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        100000 non-null  int64  
 1   region_id       100000 non-null  int64  
 2   city            100000 non-null  int64  
 3   courier_id      100000 non-null  int64  
 4   lng             100000 non-null  float64
 5   lat             100000 non-null  float64
 6   aoi_id          100000 non-null  float64
 7   aoi_type        100000 non-null  int64  
 8   pickup_gps_lng  100000 non-null  float64
 9   pickup_gps_lat  100000 non-null  float64
 10  accept_gps_lng  100000 non-null  float64
 11  accept_gps_lat  100000 non-null  float64
 12  task_duration   100000 non-null  float64
 13  distance        100000 non-null  float64
 14  ETA             100000 non-null  float64
 15  accept_hour     100000 non-null  int32  
 16  accept_day      100000 non-null  int32  
 17  accept_mo

In [49]:
pick.columns

Index(['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id',
       'aoi_type', 'pickup_gps_lng', 'pickup_gps_lat', 'accept_gps_lng',
       'accept_gps_lat', 'task_duration', 'distance', 'ETA', 'accept_hour',
       'accept_day', 'accept_month', 'accept_weekday', 'is_weekend'],
      dtype='object')

In [50]:
numerical_features=['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat',
       'aoi_type', 'pickup_gps_lng', 'pickup_gps_lat', 'accept_gps_lng',
       'accept_gps_lat', 'accept_hour','accept_day', 'accept_month', 'accept_weekday', 'is_weekend']

In [ ]:
scaler_svr = StandardScaler()
X_train_svr = X_train.copy()
X_valid_svr = X_valid.copy()
X_test_svr = X_test.copy()

# Scale only numerical features
X_train_svr[numerical_features] = scaler_svr.fit_transform(X_train_svr[numerical_features])
X_valid_svr[numerical_features] = scaler_svr.transform(X_valid_svr[numerical_features])
X_test_svr[numerical_features] = scaler_svr.transform(X_test_svr[numerical_features])

In [52]:
# Initialize and Train SVR with RBF Kernel
svr = SVR(kernel='rbf', C=10, gamma='scale')
svr.fit(X_train_svr, y_train)

# Predictions
y_pred_svr = svr.predict(X_test_svr)

# Evaluation Metrics for SVR
mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = mse_svr ** 0.5
r2_svr = r2_score(y_test, y_pred_svr)

print("\nSupport Vector Regression (RBF Kernel) Performance:")
print(f"MAE: {mae_svr}")
print(f"MSE: {mse_svr}")
print(f"RMSE: {rmse_svr}")
print(f"R² Score: {r2_svr}")


Support Vector Regression (RBF Kernel) Performance:
MAE: 0.3147111661267153
MSE: 0.15106027712039935
RMSE: 0.3886647361420886
R² Score: 0.8050439501773771


In [ ]:
# Define parameter grids
gbr_params = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

svr_params = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'epsilon': [0.01, 0.1, 0.5]
}

# Gradient Boosting Regressor Tuning
gbr = GradientBoostingRegressor()
gbr_grid = GridSearchCV(gbr, gbr_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
gbr_grid.fit(X_train, y_train)

# Support Vector Regression Tuning
svr = SVR(kernel='rbf')
svr_grid = GridSearchCV(svr, svr_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
svr_grid.fit(X_train_svr, y_train)  

# Best models
best_gbr = gbr_grid.best_estimator_
best_svr = svr_grid.best_estimator_

print("Best GBR Params:", gbr_grid.best_params_)
print("Best SVR Params:", svr_grid.best_params_)

Best GBR Params: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
Best SVR Params: {'C': 1, 'epsilon': 0.01, 'gamma': 0.01}


In [54]:
# Predict on Test Set
y_pred_gbr = best_gbr.predict(X_test)
y_pred_svr = best_svr.predict(X_test_svr)

# Evaluation Metrics
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"{model_name} Performance:")
    print(f"MAE: {mae}")
    print(f"MSE: {mse}")
    print(f"RMSE: {rmse}")
    print(f"R² Score: {r2}")
    print("-" * 40)

# Evaluate both models
evaluate_model(y_test, y_pred_gbr, "Gradient Boosting Regressor")
evaluate_model(y_test, y_pred_svr, "Support Vector Regression")

Gradient Boosting Regressor Performance:
MAE: 8.034843622825065e-09
MSE: 1.5389621957769553e-16
RMSE: 1.2405491508912314e-08
R² Score: 0.9999999999999998
----------------------------------------
Support Vector Regression Performance:
MAE: 0.06562955951849063
MSE: 0.006778331853226874
RMSE: 0.08233062524496504
R² Score: 0.991251990081822
----------------------------------------


In [57]:
#dump the model
joblib.dump(best_gbr, 'best_eta_model.pkl')

['best_eta_model.pkl']

Gradient Boosting Regressor performs better with a MAE of 8.03e-09, MSE of 1.54e-16, RMSE of 1.24e-08, and R² of 0.9999999999999998, compared to SVR’s MAE of 0.0656, MSE of 0.00678, RMSE of 0.0823, and R² of 0.9912.